In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import sys, os
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from collections import defaultdict
from scipy.stats import gaussian_kde
sys.path.append(os.path.abspath(".."))
from models.utils_common import population as pop
import models.Burkart2022.model.PAF_calculations as paf
import utils

### Data analysis

In [ ]:
### AR6 pathways
ar6 = pd.read_csv('X:\\user\\dekkerm\\Data\\AR6_snapshots\\AR6_snapshot_Nayeli_global.csv')
cc_path_mean = ar6.iloc[:, 10:-1].groupby(ar6['Category']).mean()
cc_path_std = ar6.iloc[:, 10:-1].groupby(ar6['Category']).std()

original_colors = ['C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7']
fig, ax = plt.subplots(figsize=(8, 5))
for i,categories in enumerate(ar6['Category'].unique()):
    plt.plot(cc_path_mean.columns, cc_path_mean.loc[categories], label=categories, color=original_colors[i])
    plt.fill_between(cc_path_mean.columns, cc_path_mean.loc[categories] - 2*cc_path_std.loc[categories], cc_path_mean.loc[categories] + 2*cc_path_std.loc[categories], 
                     color=original_colors[i], alpha=0.2)
ax.set_xticks(cc_path_mean.columns[::10])
utils.StylizePlot(title='CC pathways - AR6 \n(2std uncertainty range)', ylabel='GMST anomaly (°C)', xlabel='Year', legend=True, show=True)

#### Temperature Zones

In [ ]:
### Map of the temperature zones

wdir = "X:/user/liprandicn/mt-comparison/burkart2022"
era5_tz = xr.open_dataset(f"{wdir}/data/temperature_zones/ERA5_mean_1980-2019_land_t2m_tz.nc")
fig = plt.figure(figsize=(10,5), dpi=300)
ax = fig.add_subplot(111, projection=ccrs.PlateCarree(central_longitude=0), frameon=False)
ax.coastlines(resolution='10m', lw = 0.1)
era5_tz.t2m.plot(ax=ax, transform=ccrs.PlateCarree(), cbar_kwargs = {'orientation': 'horizontal', 'shrink':0.6, 'pad':0.05, 'aspect':20, 'label':'Temperature zones'},
                 vmin=6, vmax=28, cmap = 'cividis')

#### TMRELs

##### Population weighted PDFs of TMRELs vs Temperature Zones

In [ ]:
### Open all files and calculate KDE

wdir = "X:/user/liprandicn/mt-comparison/burkart2022"
years = [1990,2010,2020]

# Import population maps for years with TMRELs
population = pop.LoadPopulationMap(wdir, scenario="SSP2_ERA5", ssp="SSP2", years=[1990,2020])
population = population.sel(time=["1990", "2010", "2020"]).mean("time")

# Import TMRELs maps
tmrel = {}
for year in years:
    tmrel[year] = xr.open_dataset(wdir+f"/data/TMRELs_nc/TMRELs_{year}.nc")

# Import temperature zones
era5_tz = xr.open_dataset(wdir+f"/data/temperature_zones/ERA5_mean_1980-2019_land_t2m_tz.nc")

# Initialize dicitonaries for counts and weights for KDE
counts_tmrel = defaultdict(dict); weights_tmrel = defaultdict(dict)

for year in years:
    for tz in range(6,29):
        counts_tmrel[year][tz], weights_tmrel[year][tz] = utils.WeightsAndCountsForKDE(tmrel[year], era5_tz, population,tz)

In [ ]:
### Plot TMRELs vs TZ for one year
year = 2010
tz_values = list(range(6, 29))
colors = cm.copper(np.linspace(0, 1, len(tz_values)))

fig, ax = plt.subplots(figsize=(10, 8))#, dpi=300)

for i, tz in enumerate(tz_values):
    data = counts_tmrel[year][tz]
    w = weights_tmrel[year][tz]

    if len(data) > 50000:
        idx = np.random.choice(len(data), size=50000, replace=False, p=w)
        data = data[idx]
        w = w[idx]
        w = w / w.sum()

    kde = gaussian_kde(data, weights=w)

    p5, p95 = np.percentile(data, [5, 95])
    x = np.linspace(p5, p95, 200)
    y = kde(x)

    scale = 0.4 / y.max()
    y_scaled = y * scale

    ax.fill_betweenx(x, tz, tz + y_scaled, color=colors[i], alpha=0.8, zorder=2)

utils.StylizeAxes(ax, xticks=tz_values, xtickslabels=tz_values, xlim=(5.5,28.5), ylim=(14,30), xlabel='Temperature zone [°C]',
             ylabel="TMREL [°C]", title=f"Population-weighted TMREL distributions by Temperature Zone for {year}\n5–95 Percentile Range\n",
             grid=False)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

In [ ]:
### Plot TMRELs vs TZ for all years

tz_values = list(range(6, 29))
colors = cm.copper(np.linspace(0, 1, len(tz_values)))

fig, axs = plt.subplots(1,3, figsize=(18, 5))#, dpi=300)
axs=axs.flatten()

for j,year in enumerate([1990,2010,2020]):
    for i, tz in enumerate(tz_values):
        data = counts_tmrel[year][tz]
        w = weights_tmrel[year][tz]

        if len(data) > 50000:
            idx = np.random.choice(len(data), size=50000, replace=False, p=w)
            data = data[idx]
            w = w[idx]
            w = w / w.sum()

        kde = gaussian_kde(data, weights=w)

        p5, p95 = np.percentile(data, [5, 95])
        x = np.linspace(p5, p95, 200)
        y = kde(x)

        scale = 0.4 / y.max()
        y_scaled = y * scale

        axs[j].fill_betweenx(x, tz, tz + y_scaled, color=colors[i], alpha=0.9, zorder=2)

    utils.StylizeAxes(axs[j], xticks=tz_values, xtickslabels=tz_values, xlim=(5.5,28.5), ylim=(13.5,30), xlabel='Temperature zone [°C]',
                ylabel="TMREL [°C]", title=f"{year}", grid=True, grid_kwargs={'axis':'both', 'color':'white', 'zorder':0}, facecolor='whitesmoke')
plt.suptitle('Population-weighted TMREL Distributions by Temperature Zone')

#### ERFs

In [ ]:
### Plotting relevant causes mean of RR

sets = paf.ModelSettings(
        wdir="X:/user/liprandicn/mt-comparison/burkart2022",
        temp_dir="X:/user/liprandicn/Data/ERA5/t2m_daily",
        project=None,
        scenario="SSP2_ERA5",
        years=[2010],
        draw="mean",
        single_erf=False, 
        extrap_erf=False,
    )

erf, _, _ = paf.LoadExposureResponseFunctions(sets)

plt.rcParams["axes.prop_cycle"] = plt.cycler("color", plt.cm.magma(np.linspace(0,1,23)))

fig, axs = plt.subplots(2,4, figsize=(18,8))#, dpi=300)
axs = axs.flatten()

for i, disease in enumerate(utils.relevant_causes.keys()):
    for tz in range(6,29):
        
        x = erf["daily_temperature"].loc[tz].values   # Daily Temperature
        y = erf[disease].loc[tz].values    # Relative Risk
        
        # x_shift = x - x[np.argmin(y)] # Horizontal shift
        y_shift = y - np.min(y) + 1   # Move the minimum to 1 RR
        
        axs[i].plot(x, y_shift, label=tz)

    utils.StylizeAxes(axs[i], facecolor='whitesmoke', grid=True, grid_kwargs={'c':'white'}, title=f'{utils.relevant_causes[disease]}')#, ylim=(0.9,2))
    
    for i in [0,4]:
        utils.StylizeAxes(axs[i], ylabel='Relative Risk')
    for i in range(4,8):
        utils.StylizeAxes(axs[i], xlabel='Daily temperature [°C]')
    
plt.legend(loc='upper center', bbox_to_anchor=(-1.3, -0.2), ncol=8, title = 'Temperature zones')

In [ ]:
### Plotting EXTRAPOLATION of ERFs
sets = paf.ModelSettings(
        wdir="X:/user/liprandicn/mt-comparison/burkart2022",
        temp_dir="X:/user/liprandicn/Data/ERA5/t2m_daily",
        project=None,
        scenario="SSP2_ERA5",
        years=[2010],
        draw="mean",
        single_erf="False", 
        extrap_erf=True,
    )

erf_ex, _, _ = paf.LoadExposureResponseFunctions(sets)

plt.rcParams["axes.prop_cycle"] = plt.cycler("color", plt.cm.magma(np.linspace(0,1,23)))

fig, axs = plt.subplots(2,4, figsize=(18,8))#, dpi=300)
axs = axs.flatten()

for i, disease in enumerate(utils.relevant_causes.keys()):
    for tz in range(6,29):
        
        # --- Original data ---
        x_orig = erf["daily_temperature"].loc[tz].values
        y_orig = erf[disease].loc[tz].values
        y_orig_shift = y_orig - np.min(y_orig) + 1  # Vertical shift
        x_orig_shift = x_orig - x_orig[np.argmin(y_orig)] # Horizontal shift
        axs[i].plot(x_orig_shift, y_orig_shift, label=tz, linestyle='-')  # Solid line

        # --- Extrapolated data ---
        x_ex = erf_ex["daily_temperature"].loc[tz].values
        y_ex = erf_ex[disease].loc[tz].values
        y_ex_shift = y_ex - np.min(y_ex) + 1  # Vertical shift
        x_ex_shift = x_ex - x_ex[np.argmin(y_ex)] # Horizontal shift
        axs[i].plot(x_ex_shift, y_ex_shift, linestyle='--')  # Dashed line

    utils.StylizeAxes(axs[i], facecolor='whitesmoke', grid=True, grid_kwargs={'c':'white'}, title=f'{utils.relevant_causes[disease]}', ylim=(0.9,5))
    
    for i in [0,4]:
        utils.StylizeAxes(axs[i], ylabel='Relative Risk')
    for i in range(4,8):
        utils.StylizeAxes(axs[i], xlabel='Daily temperature [°C]')
    
plt.legend(loc='upper center', bbox_to_anchor=(-1.3, -0.2), ncol=8, title = 'Temperature zones')